## 4. Forward PINNs

**Question.** If a small neural network is given only the ODE and the initial condition, how well does it predict the population, compared with a standard solver? All data are *synthetic and dimensionless* ($x=N/K$, $\tau=rt$). The domain is $\tau\in[0,T]$ ($T=10$; for Model A, $T$ is the extinction time) and $x\in[0,1]$. The setting below is the **same one used for the inverse problem** (Section 5).

**Equations.** Model A: $\dot x=G(x)-q$; Model B: $\dot x=G(x)-ex$; $G(x)=x(1-x)(x/a-1)$. Write $\dot x=f(x)$.

**Network with a hard initial condition.** A fully connected network $N(s;\theta)$ with layer sizes $(1,32,32,1)$ (JAX, float64) and the $\tanh$ activation, with fixed time scaling $s=\tau/T$, defines

$$\hat x(\tau)=x_0+s\,N(s;\theta),$$

so $\hat x(0)=x_0$ **exactly for any weights**: no initial-condition loss is needed. Automatic differentiation with respect to physical time applies the chain rule $d\hat x/d\tau=T^{-1}\,d\hat x/ds$ (with the product rule on $s\,N$).

**Loss.** At $n=300$ fixed uniform collocation times $\tau_i\in[0,T]$:

$$R(\tau)=\frac{d\hat x}{d\tau}-f(\hat x),\qquad \mathcal L(\theta)=\frac1n\sum_i R(\tau_i)^2 .$$

**Algorithm.** (1) fixed collocation grid; (2) $\hat x$ and $d\hat x/d\tau$ by `jax.grad` + `vmap`; (3) $\mathcal L$; (4) Adam for 1000 iterations at learning rate $10^{-3}$ (about $2/7$ of the 3500-iteration budget) to reach a good region, then L-BFGS for 2500 iterations with the tolerance stop switched off, to converge. No solver solution is used as a label. For A the training interval ends at the extinction time, so the zero tail never enters the residual.

**Evaluation.** Dense DOP853 reference (rtol $10^{-12}$), separate from training. Metrics: RMSE, MAE, global relative $L^2$ (reported in scientific notation), max error, max error where $x_{\rm ref}<0.05$ (near-zero error), negative samples, runtime. A prediction containing NaN or infinity is *never scored* (it is labelled invalid and cannot count as accurate); a run that raises an exception is recorded as failed. A run is *inaccurate* if relative $L^2>5\times10^{-2}$ (threshold fixed beforehand).


In [ ]:
# Live demo (a few seconds on CPU): short run (100 Adam + 500 L-BFGS), Model B regular, seed 0, final design.
# A short budget is not the final result; compare with the curated table below.
import sys

sys.path.insert(0, "experiments/wp04")
import run as wp04

from mm_allee.pinn.train_forward import evaluate_prediction, predict

case = dict(model="B", a=0.2, harvest=0.3, x0=0.8, tau_end=10.0)
ref = dict(method="DOP853", rtol=1e-12, atol=1e-14, n_points=1001)
tau, x_ref, hit = wp04.reference_trajectory(case, ref)
training = {
    "activation": "tanh",
    "lr": 1e-3,
    "n_collocation": 300,
    "adam_iters": 100,
    "lbfgs_iters": 500,
}
res = wp04.run_pinn(case, 0, {"training": training}, hit)
ev = evaluate_prediction(predict(res.params, res.config, tau), x_ref)
print({k: (f"{v:.3e}" if isinstance(v, float) else v) for k, v in ev.items()})
print(f"{res.wall_seconds:.1f} s, final physics loss {res.history['loss'][-1]:.3e}")

## How the setting was chosen

The choice was made in two steps, and the second corrected the first.

1. **Activation at learning rate $10^{-2}$.** tanh, swish and sigmoid were compared with a rule fixed beforehand (most accurate runs of 18; seeds 0/1/42): swish 17/18, tanh 11/18, sigmoid 10/18. This varied the activation at one learning rate only.
2. **Joint study with the inverse problem.** A pre-registered study varied the knobs the two problems must share and scored each candidate by *(forward runs accurate of 18) + (inverse fits recovered of 18)*. The 64-wide network was dropped after its first candidate was slower and less accurate (score 23 against 28 at width 32). The network learning rate turned out to be the dominant factor ($10^{-3}$ beats $10^{-2}$ for both activations); tanh and swish at $10^{-3}$ tie on the score, the tie-break selects tanh, and on fresh seeds the two cannot be told apart with three seeds per set (not a proof of equivalence).

Both tables are loaded from `artifacts/tables/`. *Caveat:* the selection used the same seeds as the final sweep, so its accuracy is optimistic; the fresh-seed confirmation further down is the unbiased evidence.

In [ ]:
import csv  # noqa: F811  (an earlier fragment imports csv in the assembled notebook)

print("Step 1: activation at lr 1e-2 (artifacts/tables/wp04_activation_selection.csv)")
for r in csv.DictReader(open("artifacts/tables/wp04_activation_selection.csv")):
    print(
        f"  {r['activation']:8s} accurate {r['n_accurate']:>2s}/{r['n_runs']}  geomean rel L2 {float(r['geomean_relative_l2']):.2e}"
    )
print("\nStep 2: shared-settings study, width 32 (artifacts/tables/shared_stage1_summary.csv)")
for r in csv.DictReader(open("artifacts/tables/shared_stage1_summary.csv")):
    print(
        f"  {r['candidate']:22s} forward {r['forward_accurate']:>2s}/{r['forward_runs']}  inverse {r['inverse_recovered']:>2s}/{r['inverse_fits']}  "
        f"score {r['score']:>2s}  selected: {r['selected']}"
    )

## Curated final results (loaded, not retrained)

The table is `artifacts/tables/wp04_forward_metrics.csv`, produced by
`python experiments/wp04/run.py --config configs/wp04_final.json --output results/wp04 --artifacts artifacts` (about 1.2 min on CPU; see `artifacts/manifests/wp04_forward.json` and the pinned versions in `wp04_forward_requirements.txt`).


In [ ]:
rows = list(csv.DictReader(open("artifacts/tables/wp04_forward_metrics.csv")))


def fmt(v):
    return f"{float(v):.2e}" if v else "-"


print(
    f"{'case':13s}{'method':7s}{'seed':5s}{'status':12s}{'rel L2':>10s}{'max err':>10s}{'near-0':>10s}{'neg':>5s}{'sec':>7s}"
)
for r in rows:
    print(
        f"{r['case']:13s}{r['method']:7s}{r['seed']:5s}{r['status']:12s}{fmt(r['relative_l2']):>10s}"
        f"{fmt(r['max_abs']):>10s}{fmt(r['max_abs_near_zero']):>10s}{r['boundary_violations']:>5s}{float(r['wall_seconds']):7.2f}"
    )

![Forward PINN vs reference](../artifacts/figures/wp04_forward.png)

![Error and loss](../artifacts/figures/wp04_error_loss.png)

Curated PNGs from the final run (EPS and PDF copies are in `artifacts/figures/`). The vertical line in the bottom row marks the Adam to L-BFGS hand-over.

## Interpretation

- **Final sweep (tanh, lr $10^{-3}$, seeds 0/1/42):** all 18 runs are accurate, relative $L^2$ from $2.9\times10^{-7}$ to $4.1\times10^{-3}$ (median $6.4\times10^{-6}$); the largest error is A near-fold (collapse after a bottleneck), the hardest case.
- Five runs predict slightly negative populations near zero (B decay seeds 1 and 42: 23 and 61 of 1001 samples; A extinction and A near-fold: 1 sample each). Near-zero error (largest absolute error where $x_{\rm ref}<0.05$): A extinction $8.9\times10^{-8}$ to $1.9\times10^{-7}$, B decay $7.3\times10^{-7}$ to $2.6\times10^{-6}$, A near-fold $4.8\times10^{-6}$ to $8.6\times10^{-3}$.
- In every run L-BFGS stopped at its 2500-iteration limit (not an abnormal exit), so the iteration budget is binding; longer runs were not tried.
- RK45 matches the reference to about $10^{-10}$ in $4$ to $10$ ms; each PINN run took $3.2$ to $4.6$ s. For a scalar ODE the classical solver is the better tool; the PINN's value is as the engine for the inverse problem (Section 5).
- For Model A the horizon $T$ is the extinction time from the reference solver: the forward PINN does **not** predict the extinction time itself.
- *Limitations:* one parameter set per regime, three seeds, setting selected on the same seeds (then confirmed on three others, below), one machine; per-seed results depend on platform and library versions (exact versions are in the manifests). A **small training loss is not evidence of accuracy**: see the diagnostic below.


## Held-out confirmation and reproducibility

The frozen design (tanh, $(1,32,32,1)$, lr $10^{-3}$, 300 points) was run unchanged on seeds 11, 22, 33, which were **not** used to select it. Table loaded from `artifacts/tables/wp04_confirmation_metrics.csv` (about 1.2 min on CPU to regenerate with `configs/wp04_confirm.json`). All 18 runs are accurate (A near-fold $1.8\times10^{-4}$ to $2.3\times10^{-3}$ is the largest); both seed sets give **36 of 36 accurate runs**. Reviewers' independent reruns on Linux with newer libraries of the *earlier* swish design (lr $10^{-2}$) reproduced the counts but failed a different A near-fold seed, so per-seed results depend on platform and library versions; the final tanh design has not yet been rerun by another member. The exact versions used here are in `artifacts/manifests/wp04_*_requirements.txt`.

In [ ]:
conf = [
    r
    for r in csv.DictReader(open("artifacts/tables/wp04_confirmation_metrics.csv"))
    if r["method"] == "pinn"
]
for case in dict.fromkeys(r["case"] for r in conf):
    errs = [float(r["relative_l2"]) for r in conf if r["case"] == case]
    print(f"{case:13s} seeds 11/22/33 rel L2: " + ", ".join(f"{e:.1e}" for e in errs))
print("accurate:", sum(float(r["relative_l2"]) <= 5e-2 for r in conf), "of", len(conf))

## Diagnostic: why did the regular cases fail with tanh, 100 points and lr $10^{-2}$?

An *exploratory* experiment (`experiments/wp04/diagnose.py`; hypotheses were written in `experiments/wp04/README.md` before running) on the **earlier** design (tanh, 100 uniform points, learning rate $10^{-2}$, seeds 11/22/33), separate from the final sweep. A regular and B regular are trained with 10 seeds under five collocation designs: `U100` (the earlier design), `U200`, `U400` (denser uniform), `P100` (100 points clustered near $\tau=0$, $\tau_i=T(i/99)^2$) and `U100_long` (10000 L-BFGS iterations). Besides the error we measure the residual $|R|$ on a dense grid *between* the training points. Command: `python experiments/wp04/diagnose.py --config configs/wp04_diagnose.json --output results/wp04_diagnose --artifacts artifacts` (about 8 min on CPU); the table below is loaded, not recomputed.


In [ ]:
for r in csv.DictReader(open("artifacts/tables/wp04_diagnose_summary.csv")):
    med = float(r["median_relative_l2"])
    print(
        f"{r['case']:10s}{r['variant']:10s} accurate {r['n_accurate']:>2s}/10  unstable-root {r['n_unstable_root']:>2s}  "
        f"zero {r['n_zero']:>2s}  negative {r['n_negative_state']:>2s}  not-converged {r['n_not_converged']:>2s}  "
        f"other {r['n_other']:>2s}  median rel L2 {med:.1e}"
    )

![Diagnostic outcomes](../artifacts/figures/wp04_diagnose.png)

**Findings (10 seeds per cell, tanh, lr $10^{-2}$).**

- **Mechanism:** every accurate run has max dense-grid residual $\le1.2\times10^{-3}$ and every failed run $\ge8.4\times10^{-1}$, yet 45 of the 55 failures had residual $<10^{-1}$ *at the training points*. The loss at the training points cannot reveal them; a residual check on a denser grid separates all 100 runs.
- **Resolution near $\tau=0$ matters, not the point count:** denser uniform grids help only a little (accurate runs: A 3, 3, 5 and B 2, 3, 4 for 100, 200, 400 points), whereas the same 100 points clustered near $\tau=0$ gave 10/10 in both cases (median relative $L^2$ $6.5\times10^{-6}$ for A, $5.3\times10^{-6}$ for B).
- **More L-BFGS iterations do not help:** 3/10 (A) and 2/10 (B) stay accurate.
- Failures end on spurious equilibria of the ODE: the unstable root (22 runs), zero (6), the unphysical negative root of the Model A polynomial (2), or elsewhere; 10 runs had not converged (all Model A, uniform grids).
- The final design (tanh, lr $10^{-3}$, 300 uniform points) does not show these failures (36 of 36 accurate on six seeds); which of the changes (learning rate, number of points) removed them was not isolated.
- *Caveats:* two cases only; the clustered grid was chosen after seeing the failures and was not adopted; 10 seeds cannot exclude a failure rate up to about $25$-$30\%$; outcomes are sensitive to round-off (a grid differing by one rounding unit changed one failing seed's error from $7.6\times10^{-1}$ to $9.8\times10^{-1}$).
